# LeetCode 1357 — Apply Discount Every n Orders bằng Python

Link đề: https://leetcode.com/problems/apply-discount-every-n-orders/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Design

Skeleton LeetCode (bài **thiết kế lớp**):

```python
class Cashier:

    def __init__(self, n: int, discount: int, products: list[int], prices: list[int]):
        ...

    def getBill(self, product: list[int], amount: list[int]) -> float:
        ...
```

## 1. Mục tiêu học

1. Đọc hiểu một bài design "mô phỏng nghiệp vụ": lưu **trạng thái** (số khách đã thanh toán) giữa các lần gọi phương thức.
2. Chuyển hai mảng song song `products`/`prices` thành **bảng tra cứu** `id → giá` (dict hoặc mảng chỉ số) để tính hóa đơn nhanh.
3. Dùng phép chia lấy dư `count % n == 0` để nhận diện **khách thứ n, 2n, 3n, …**.
4. Hiểu **số thực** và sai số: `bill * (100 − d) / 100`, so sánh trong phạm vi `10^-5`, kiểm chứng bằng `fractions.Fraction`.
5. Viết **bộ giả lập chuỗi lệnh** và đối chiếu nhiều cài đặt.

## 2. Đề bài & ví dụ

Siêu thị có các sản phẩm với mã `products[i]` và giá `prices[i]` (hai mảng song song). Khi một khách thanh toán, hóa đơn gồm hai mảng song song `product` (mã sản phẩm) và `amount` (số lượng). **Tạm tính** `= Σ amount[j] × giá(product[j])`.

Cứ **mỗi khách thứ `n`** (khách thứ `n`, `2n`, `3n`, …) được giảm `discount` phần trăm: trả `bill × (100 − discount) / 100`.

Cài đặt lớp `Cashier`:
- `Cashier(n, discount, products, prices)` khởi tạo.
- `getBill(product, amount)` trả về tổng tiền sau giảm giá (nếu có). Sai số `10^-5` được chấp nhận.

**Ví dụ:**
```
["Cashier","getBill","getBill","getBill","getBill","getBill","getBill","getBill"]
[[3,50,[1,2,3,4,5,6,7],[100,200,300,400,300,200,100]],[[1,2],[1,2]],[[3,7],[10,10]],[[1,2,3,4,5,6,7],[1,1,1,1,1,1,1]],[[4],[10]],[[7,3],[10,10]],[[7,5,3,1,6,4,2],[10,10,10,9,9,9,7]],[[2,3,5],[5,3,2]]]
Output: [null,500.0,4000.0,800.0,4000.0,4000.0,7350.0,2500.0]
```
Khách 3 và 6 được giảm 50%: `1600 → 800`, `14700 → 7350`.

**Ràng buộc chính:** `1 <= n <= 10^4`; `0 <= discount <= 100`; `1 <= products.length <= 200`; `1 <= products[i] <= 200` (đôi một khác nhau); `1 <= prices[i] <= 1000`; `1 <= amount[j] <= 1000`; mã trong `product` đôi một khác nhau và luôn có trong `products`; tối đa `1000` lời gọi `getBill`.

In [1]:
import random
import time
from fractions import Fraction

EX_OPS = ["Cashier", "getBill", "getBill", "getBill", "getBill", "getBill", "getBill", "getBill"]
EX_ARGS = [[3, 50, [1, 2, 3, 4, 5, 6, 7], [100, 200, 300, 400, 300, 200, 100]],
           [[1, 2], [1, 2]], [[3, 7], [10, 10]], [[1, 2, 3, 4, 5, 6, 7], [1, 1, 1, 1, 1, 1, 1]],
           [[4], [10]], [[7, 3], [10, 10]], [[7, 5, 3, 1, 6, 4, 2], [10, 10, 10, 9, 9, 9, 7]], [[2, 3, 5], [5, 3, 2]]]
EX_OUT = [None, 500.0, 4000.0, 800.0, 4000.0, 4000.0, 7350.0, 2500.0]

def run(cls, ops, args):
    """Bộ GIẢ LẬP kiểu LeetCode."""
    obj = cls(*args[0])
    out = [None]
    for op, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, op)(*a))
    return out

def same(out, want, eps=1e-5):
    """So sánh output có số thực trong phạm vi sai số."""
    if len(out) != len(want):
        return False
    return all((a is None and b is None) or (a is not None and b is not None and abs(a - b) <= eps)
               for a, b in zip(out, want))

print("Output mong đợi:", EX_OUT)

Output mong đợi: [None, 500.0, 4000.0, 800.0, 4000.0, 4000.0, 7350.0, 2500.0]


## 3. Kiến thức nền

### 3.1 Lớp có trạng thái

Thuộc tính `self.count` sống qua nhiều lần gọi `getBill` — đó là "bộ nhớ" của máy tính tiền. Hàm `run` ở trên tạo **một** đối tượng và gọi các phương thức lần lượt trên nó.

In [2]:
class Counter1:
    def __init__(self):
        self.count = 0
    def hit(self):
        self.count += 1
        return self.count

print(run(Counter1, ["Counter1", "hit", "hit", "hit"], [[], [], [], []]))

[None, 1, 2, 3]


### 3.2 `zip` và `dict(zip(...))` — biến hai mảng song song thành bảng tra

`zip(products, prices)` ghép từng cặp `(mã, giá)`; `dict(...)` tạo bảng `mã → giá`, tra cứu `O(1)`.

In [3]:
products, prices = [1, 2, 3, 7], [100, 200, 300, 100]
price = dict(zip(products, prices))
print(price, "| giá của mã 7:", price[7])
print("tạm tính:", sum(a * price[p] for p, a in zip([3, 7], [10, 10])))

{1: 100, 2: 200, 3: 300, 7: 100} | giá của mã 7: 100
tạm tính: 4000


### 3.3 Mảng chỉ số thay cho dict

Vì mã sản phẩm `≤ 200`, có thể dùng list `price_arr` dài 201 với `price_arr[mã] = giá`. Tra cứu bằng chỉ số còn nhanh hơn dict một chút.

In [4]:
price_arr = [0] * 201
for p, c in zip(products, prices):
    price_arr[p] = c
print(price_arr[:8], "| giá mã 3:", price_arr[3])

[0, 100, 200, 300, 0, 0, 0, 100] | giá mã 3: 300


### 3.4 Phép chia lấy dư — "mỗi khách thứ n"

Đánh số khách từ 1. Khách thứ `k` được giảm ⇔ `k % n == 0`. Chú ý phải **tăng bộ đếm trước** rồi mới kiểm tra.

In [5]:
n = 3
print([k for k in range(1, 11) if k % n == 0], "<- các khách được giảm khi n = 3")

[3, 6, 9] <- các khách được giảm khi n = 3


### 3.5 Số thực, sai số, và `Fraction`

`bill * (100 − d) / 100` là `float`. Các cách viết khác nhau (`bill * (1 − d/100)`, `bill − bill*d/100`) có thể lệch nhau ở chữ số rất nhỏ — vẫn trong phạm vi `10^-5`. `Fraction` cho kết quả **chính xác** để kiểm chứng.

In [6]:
bill, d = 14703, 33
a = bill * (100 - d) / 100
b = bill * (1 - d / 100)
c = bill - bill * d / 100
exact = Fraction(bill) * (100 - d) / 100
print(repr(a), repr(b), repr(c))
print("chính xác:", exact, "=", float(exact))
print("lệch tối đa so với chính xác:", max(abs(Fraction(x) - exact) for x in (a, b, c)) < Fraction(1, 10**5))

9851.01 9851.009999999998 9851.01
chính xác: 985101/100 = 9851.01
lệch tối đa so với chính xác: True


### 3.6 `sum` với generator

`sum(a * price[p] for p, a in zip(product, amount))` — gọn, không tạo list trung gian.

In [7]:
print(sum(x * y for x, y in zip([1, 2, 3], [10, 20, 30])))

140


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ (`n = 3`, `discount = 50`). Bảng giá: `1→100, 2→200, 3→300, 4→400, 5→300, 6→200, 7→100`.

| Khách | Hóa đơn | Tạm tính | k % 3 == 0? | Phải trả |
|---|---|---|---|---|
| 1 | 1×100 + 2×200 | 500 | không | 500 |
| 2 | 10×300 + 10×100 | 4000 | không | 4000 |
| 3 | 100+200+…+100 | 1600 | **có** | 800 |
| 4 | 10×400 | 4000 | không | 4000 |
| 5 | 10×100 + 10×300 | 4000 | không | 4000 |
| 6 | 1000+3000+3000+900+1800+3600+1400 | 14700 | **có** | 7350 |
| 7 | 5×200 + 3×300 + 2×300 | 2500 | không | 2500 |

**Trực giác:** chỉ cần (1) bảng giá tra nhanh, (2) một biến đếm khách.

In [8]:
price = dict(zip(EX_ARGS[0][2], EX_ARGS[0][3]))
for k, (prod_, amt) in enumerate(EX_ARGS[1:], start=1):
    sub = sum(a * price[p] for p, a in zip(prod_, amt))
    pay = sub * (100 - 50) / 100 if k % 3 == 0 else sub
    print(f"khách {k}: tạm tính {sub:6d} -> trả {pay}")

khách 1: tạm tính    500 -> trả 500
khách 2: tạm tính   4000 -> trả 4000
khách 3: tạm tính   1600 -> trả 800.0
khách 4: tạm tính   4000 -> trả 4000
khách 5: tạm tính   4000 -> trả 4000
khách 6: tạm tính  14700 -> trả 7350.0
khách 7: tạm tính   2500 -> trả 2500


## 5. Cách 1 — Brute Force / baseline: tìm giá bằng duyệt tuyến tính

**Ý tưởng:** giữ nguyên hai mảng `products`, `prices`. Với mỗi mục trong hóa đơn, **duyệt** `products` để tìm vị trí của mã đó rồi lấy giá. Đếm khách bằng `self.count`.

In [9]:
class CashierBrute:
    def __init__(self, n, discount, products, prices):
        self.n, self.discount = n, discount
        self.products, self.prices = products, prices
        self.count = 0

    def getBill(self, product, amount):
        self.count += 1
        bill = 0
        for p, a in zip(product, amount):
            for i in range(len(self.products)):        # tìm tuyến tính
                if self.products[i] == p:
                    bill += a * self.prices[i]
                    break
        if self.count % self.n == 0:
            bill = bill * (100 - self.discount) / 100
        return float(bill)

got = run(CashierBrute, EX_OPS, EX_ARGS)
print(got, "| đúng?", same(got, EX_OUT))

[None, 500.0, 4000.0, 800.0, 4000.0, 4000.0, 7350.0, 2500.0] | đúng? True


**Độ phức tạp:** mỗi `getBill` `O(m · P)` với `m = len(product)`, `P = len(products) ≤ 200` ⇒ xấu nhất `200 × 200 = 4·10^4` mỗi lần, `4·10^7` cho 1000 lần — chạy được nhưng thừa.

**Nút thắt:** tìm giá lặp đi lặp lại trên cùng một bảng tĩnh. Bảng giá **không đổi** ⇒ xây **bảng tra** một lần ở `__init__`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** tách hai phần việc:
1. **Tiền xử lý một lần** (trong `__init__`): `price = {mã: giá}` ⇒ mỗi lần tra `O(1)`.
2. **Trạng thái tối thiểu**: chỉ cần biết khách hiện tại là khách thứ mấy (hoặc chỉ cần `count mod n`).

**Quy tắc nếu… thì…:**
- Mỗi lần `getBill`: `count += 1` **trước**.
- Nếu `count % n == 0` thì `bill *= (100 − discount) / 100`.
- (Tùy chọn) Nếu `count == n` thì reset `count = 0` — số luôn nhỏ, điều kiện trở thành `count == 0` sau khi reset.

**Bất biến:** sau `k` lời gọi `getBill`, `self.count == k` (hoặc `k mod n` với bản reset).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Dict `mã → giá` + đếm khách

**Giải thích:** dựng `self.price = dict(zip(products, prices))`. `getBill` cộng `a * self.price[p]`, tăng đếm và áp giảm giá.

In [10]:
class CashierDict:
    def __init__(self, n, discount, products, prices):
        self.n = n
        self.discount = discount
        self.price = dict(zip(products, prices))
        self.count = 0

    def getBill(self, product, amount):
        self.count += 1
        bill = sum(a * self.price[p] for p, a in zip(product, amount))
        if self.count % self.n == 0:
            return bill * (100 - self.discount) / 100
        return float(bill)

print(same(run(CashierDict, EX_OPS, EX_ARGS), EX_OUT))

True


Code nộp LeetCode (Cách 2):

In [11]:
class Cashier:

    def __init__(self, n: int, discount: int, products: list[int], prices: list[int]):
        self.n = n
        self.discount = discount
        self.price = dict(zip(products, prices))
        self.count = 0

    def getBill(self, product: list[int], amount: list[int]) -> float:
        self.count += 1
        bill = sum(a * self.price[p] for p, a in zip(product, amount))
        if self.count % self.n == 0:
            return bill * (100 - self.discount) / 100
        return float(bill)

print(run(Cashier, EX_OPS, EX_ARGS))

[None, 500.0, 4000.0, 800.0, 4000.0, 4000.0, 7350.0, 2500.0]


**Độ phức tạp:** `__init__` `O(P)`; `getBill` `O(m)`. Bộ nhớ `O(P)`.

### 7.2 Cách 3 — Mảng chỉ số + bộ đếm vòng (reset)

**Giải thích từng đoạn:**
1. `self.price = [0] * 201`, gán theo mã — tận dụng `mã ≤ 200`.
2. Bộ đếm `self.count` tăng rồi **reset về 0** khi chạm `n` ⇒ không bao giờ vượt `n`. Giảm giá khi vừa reset.
3. Nhân hệ số `self.factor = (100 − discount) / 100` tính sẵn một lần.

In [12]:
class CashierArray:
    def __init__(self, n, discount, products, prices):
        self.n = n
        self.factor = (100 - discount) / 100
        self.price = [0] * 201
        for p, c in zip(products, prices):
            self.price[p] = c
        self.count = 0

    def getBill(self, product, amount):
        bill = 0
        for p, a in zip(product, amount):
            bill += a * self.price[p]
        self.count += 1
        if self.count == self.n:
            self.count = 0
            return bill * self.factor
        return float(bill)

print(same(run(CashierArray, EX_OPS, EX_ARGS), EX_OUT))

True


Code nộp LeetCode (Cách 3):

In [13]:
class Cashier:

    def __init__(self, n: int, discount: int, products: list[int], prices: list[int]):
        self.n = n
        self.factor = (100 - discount) / 100
        self.price = [0] * 201
        for p, c in zip(products, prices):
            self.price[p] = c
        self.count = 0

    def getBill(self, product: list[int], amount: list[int]) -> float:
        bill = 0
        for p, a in zip(product, amount):
            bill += a * self.price[p]
        self.count += 1
        if self.count == self.n:
            self.count = 0
            return bill * self.factor
        return float(bill)

print(run(Cashier, EX_OPS, EX_ARGS))

[None, 500.0, 4000.0, 800.0, 4000.0, 4000.0, 7350.0, 2500.0]


**Lưu ý nhỏ về số thực:** `bill * factor` (nhân với `0.5`, `0.67`…) và `bill * (100 − d) / 100` có thể khác nhau ở bit cuối, nhưng sai số tương đối chỉ cỡ `10^-16` — rất xa ngưỡng `10^-5`.

### 7.3 Cách 4 — Tính chính xác bằng `Fraction` (cài đặt kiểm chứng)

**Giải thích:** dùng `Fraction` cho phép giảm giá, chỉ đổi sang `float` khi trả về. Chậm hơn nhưng **không có sai số trung gian** — làm "trọng tài" trong bộ test.

In [14]:
class CashierExact:
    def __init__(self, n, discount, products, prices):
        self.n = n
        self.factor = Fraction(100 - discount, 100)
        self.price = dict(zip(products, prices))
        self.count = 0

    def getBill(self, product, amount):
        self.count += 1
        bill = Fraction(sum(a * self.price[p] for p, a in zip(product, amount)))
        if self.count % self.n == 0:
            bill *= self.factor
        return bill            # trả Fraction để so sánh chính xác

ex = run(CashierExact, EX_OPS, EX_ARGS)
print(ex)
print([None if x is None else float(x) for x in ex])

[None, Fraction(500, 1), Fraction(4000, 1), Fraction(800, 1), Fraction(4000, 1), Fraction(4000, 1), Fraction(7350, 1), Fraction(2500, 1)]
[None, 500.0, 4000.0, 800.0, 4000.0, 4000.0, 7350.0, 2500.0]


## 8. TRACE từng bước

Trace Cách 3 trên ví dụ đề: in bộ đếm, tạm tính, có giảm giá hay không.

| Lệnh | count trước | count sau | reset? | Tạm tính | Trả |
|---|---|---|---|---|---|
| bill 1 | 0 | 1 | | 500 | 500.0 |
| bill 2 | 1 | 2 | | 4000 | 4000.0 |
| bill 3 | 2 | 3 → 0 | ✔ | 1600 | 800.0 |
| bill 4 | 0 | 1 | | 4000 | 4000.0 |
| bill 5 | 1 | 2 | | 4000 | 4000.0 |
| bill 6 | 2 | 3 → 0 | ✔ | 14700 | 7350.0 |
| bill 7 | 0 | 1 | | 2500 | 2500.0 |

In [15]:
class CashierTrace(CashierArray):
    def getBill(self, product, amount):
        before = self.count
        items = [(p, a, self.price[p]) for p, a in zip(product, amount)]
        sub = sum(a * c for _, a, c in items)
        res = super().getBill(product, amount)
        disc = res != sub
        print(f"count {before} -> {self.count}{' (reset)' if disc else ''} | tạm tính {sub:6d} | trả {res}")
        return res

def trace_run(ops, args):
    return run(CashierTrace, ops, args)

out = trace_run(EX_OPS, EX_ARGS)

count 0 -> 1 | tạm tính    500 | trả 500.0
count 1 -> 2 | tạm tính   4000 | trả 4000.0
count 2 -> 0 (reset) | tạm tính   1600 | trả 800.0
count 0 -> 1 | tạm tính   4000 | trả 4000.0
count 1 -> 2 | tạm tính   4000 | trả 4000.0
count 2 -> 0 (reset) | tạm tính  14700 | trả 7350.0
count 0 -> 1 | tạm tính   2500 | trả 2500.0


Trace chi tiết từng mục của hóa đơn khách thứ 6:

In [16]:
def trace_items(price, product, amount):
    total = 0
    for p, a in zip(product, amount):
        total += a * price[p]
        print(f"  mã {p}: {a:2d} × {price[p]:3d} = {a * price[p]:5d} | cộng dồn = {total}")
    return total

price = dict(zip(EX_ARGS[0][2], EX_ARGS[0][3]))
sub = trace_items(price, *EX_ARGS[6])
print("tạm tính:", sub, "-> sau giảm 50%:", sub * 50 / 100)

  mã 7: 10 × 100 =  1000 | cộng dồn = 1000
  mã 5: 10 × 300 =  3000 | cộng dồn = 4000
  mã 3: 10 × 300 =  3000 | cộng dồn = 7000
  mã 1:  9 × 100 =   900 | cộng dồn = 7900
  mã 6:  9 × 200 =  1800 | cộng dồn = 9700
  mã 4:  9 × 400 =  3600 | cộng dồn = 13300
  mã 2:  7 × 200 =  1400 | cộng dồn = 14700
tạm tính: 14700 -> sau giảm 50%: 7350.0


## 9. Code nộp LeetCode

In [17]:
class Cashier:

    def __init__(self, n: int, discount: int, products: list[int], prices: list[int]):
        self.n = n
        self.discount = discount
        self.price = dict(zip(products, prices))     # mã -> giá
        self.count = 0                               # số khách đã thanh toán

    def getBill(self, product: list[int], amount: list[int]) -> float:
        self.count += 1
        bill = sum(a * self.price[p] for p, a in zip(product, amount))
        if self.count % self.n == 0:                 # khách thứ n, 2n, 3n, ...
            return bill * (100 - self.discount) / 100
        return float(bill)

got = run(Cashier, EX_OPS, EX_ARGS)
assert same(got, EX_OUT)
print(got)

[None, 500.0, 4000.0, 800.0, 4000.0, 4000.0, 7350.0, 2500.0]


## 10. Vì sao thuật toán đúng

1. **Bảng giá:** mã sản phẩm đôi một khác nhau ⇒ `dict(zip(products, prices))` ánh xạ mỗi mã tới đúng giá của nó; mọi mã trong hóa đơn đều tồn tại (đề đảm bảo) ⇒ tra cứu không lỗi.
2. **Tạm tính:** cộng `amount[j] × giá(product[j])` cho mọi `j` — đúng định nghĩa.
3. **Bất biến đếm:** trước lời gọi thứ `k`, `count = k − 1`; ta tăng lên `k` rồi kiểm tra `k % n == 0` ⇔ `k` là bội của `n` ⇔ khách được giảm giá. (Bản reset: `count = (k − 1) mod n` trước, sau khi tăng bằng `n` ⇔ `k ≡ 0 (mod n)`.)
4. **Giảm giá:** trả `bill × (100 − discount)/100` đúng công thức; sai số `float` tương đối ~`10^-16`, với hóa đơn tối đa `200 × 1000 × 1000 = 2·10^8` thì sai số tuyệt đối ~`10^-8` < `10^-5`.

## 11. Phân tích Time / Space

Gọi `P = len(products) ≤ 200`, `m = len(product)` của một hóa đơn, `Q` = số lời gọi `getBill`.

| Cách | `__init__` | `getBill` | Space |
|---|---|---|---|
| 1. Tìm tuyến tính | `O(1)` | `O(m · P)` | `O(P)` (tham chiếu) |
| 2. Dict | `O(P)` | `O(m)` | `O(P)` |
| 3. Mảng 201 + reset | `O(P + 201)` | `O(m)` | `O(201)` |
| 4. Fraction | `O(P)` | `O(m)` + chi phí phân số | `O(P)` |

Tổng cho `Q` lời gọi: `O(P + Σ m)` với Cách 2/3.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
IMPLS = [CashierBrute, CashierDict, CashierArray, Cashier]

def check_against_exact(ops, args):
    ref = run(CashierExact, ops, args)
    for cls in IMPLS:
        out = run(cls, ops, args)
        for a, b in zip(out[1:], ref[1:]):
            assert isinstance(a, float), (cls.__name__, type(a))
            assert abs(Fraction(a) - b) <= Fraction(1, 10**5), (cls.__name__, a, b)

check_against_exact(EX_OPS, EX_ARGS)
edge = [
    # n = 1: mọi khách đều được giảm
    (["Cashier", "getBill", "getBill"], [[1, 10, [5], [3]], [[5], [7]], [[5], [1]]], [None, 18.9, 2.7]),
    # discount = 0: không đổi
    (["Cashier", "getBill", "getBill"], [[2, 0, [1], [9]], [[1], [1]], [[1], [2]]], [None, 9.0, 18.0]),
    # discount = 100: khách thứ n trả 0
    (["Cashier", "getBill", "getBill"], [[2, 100, [1], [9]], [[1], [1]], [[1], [2]]], [None, 9.0, 0.0]),
    # n lớn hơn số khách: không ai được giảm
    (["Cashier", "getBill", "getBill"], [[10**4, 50, [200], [1000]], [[200], [1000]], [[200], [1]]], [None, 10**6 * 1.0, 1000.0]),
    # giảm lẻ: 33%
    (["Cashier", "getBill"], [[1, 33, [1, 2], [7, 11]], [[2, 1], [3, 5]]], [None, (3 * 11 + 5 * 7) * 0.67]),
]
for ops, args, want in edge:
    check_against_exact(ops, args)
    for cls in IMPLS:
        assert same(run(cls, ops, args), want), (cls.__name__, run(cls, ops, args), want)
print("Ví dụ đề + 5 kịch bản biên: mọi cài đặt đúng (so với Fraction và kết quả mong đợi)")

Ví dụ đề + 5 kịch bản biên: mọi cài đặt đúng (so với Fraction và kết quả mong đợi)


### 12.2 Đối chiếu ngẫu nhiên với bản chính xác `Fraction` (seed cố định)

In [19]:
def random_case(rng, q):
    P = rng.randint(1, 30)
    products = rng.sample(range(1, 201), P)
    prices = [rng.randint(1, 1000) for _ in range(P)]
    n = rng.randint(1, 6)
    discount = rng.randint(0, 100)
    ops, args = ["Cashier"], [[n, discount, products, prices]]
    for _ in range(q):
        m = rng.randint(1, P)
        prod_ = rng.sample(products, m)
        amt = [rng.randint(1, 1000) for _ in range(m)]
        ops.append("getBill"); args.append([prod_, amt])
    return ops, args

rng = random.Random(1357)
for _ in range(300):
    ops, args = random_case(rng, rng.randint(1, 25))
    check_against_exact(ops, args)
print("300 kịch bản ngẫu nhiên: sai số mọi cài đặt <= 1e-5 so với Fraction")

300 kịch bản ngẫu nhiên: sai số mọi cài đặt <= 1e-5 so với Fraction


### 12.3 Giới hạn: 200 sản phẩm, 1000 lời gọi, mỗi hóa đơn 200 mục

In [20]:
rng = random.Random(9)
products = list(range(1, 201))
prices = [rng.randint(1, 1000) for _ in range(200)]
ops = ["Cashier"] + ["getBill"] * 1000
args = [[7, 37, products, prices]] + [[rng.sample(products, 200), [rng.randint(1, 1000) for _ in range(200)]] for _ in range(1000)]
res = {}
for cls in IMPLS:
    st = time.perf_counter()
    res[cls.__name__] = run(cls, ops, args)
    print(f"{cls.__name__:13} {time.perf_counter() - st:.3f}s")
base = res["CashierExact"] if "CashierExact" in res else run(CashierExact, ops, args)
for name, out in res.items():
    assert all(abs(Fraction(a) - b) <= Fraction(1, 10**5) for a, b in zip(out[1:], base[1:])), name
print("Kết quả khớp bản chính xác; hóa đơn lớn nhất:", max(base[1:]))

CashierBrute  0.437s
CashierDict   0.015s
CashierArray  0.012s
Cashier       0.014s
Kết quả khớp bản chính xác; hóa đơn lớn nhất: 55060055


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra giảm giá **trước** khi tăng bộ đếm

In [21]:
class WrongOrder(CashierDict):
    def getBill(self, product, amount):
        bill = sum(a * self.price[p] for p, a in zip(product, amount))
        disc = self.count % self.n == 0          # count còn là số khách TRƯỚC đó
        self.count += 1
        return bill * (100 - self.discount) / 100 if disc else float(bill)

print("sai :", run(WrongOrder, EX_OPS, EX_ARGS))
print("đúng:", EX_OUT)

sai : [None, 250.0, 4000.0, 1600.0, 2000.0, 4000.0, 14700.0, 1250.0]
đúng: [None, 500.0, 4000.0, 800.0, 4000.0, 4000.0, 7350.0, 2500.0]


**Vì sao sai:** khách đầu tiên (`count = 0`) bị coi là "bội của n" ⇒ được giảm; khách thứ 3 lại không. Lệch một vị trí. **Sửa:** `count += 1` trước, rồi kiểm tra.

### Lỗi 2 — Dùng chia nguyên `//`

In [22]:
class WrongFloorDiv(CashierDict):
    def getBill(self, product, amount):
        self.count += 1
        bill = sum(a * self.price[p] for p, a in zip(product, amount))
        if self.count % self.n == 0:
            return bill * ((100 - self.discount) // 100)   # (100-50)//100 = 0
        return float(bill)

print("sai :", run(WrongFloorDiv, EX_OPS, EX_ARGS))
print("đúng:", EX_OUT)

sai : [None, 500.0, 4000.0, 0, 4000.0, 4000.0, 0, 2500.0]
đúng: [None, 500.0, 4000.0, 800.0, 4000.0, 4000.0, 7350.0, 2500.0]


**Vì sao sai:** `(100 − 50) // 100 = 0` ⇒ khách được giảm trả 0 đồng. Kể cả viết `bill * (100 − d) // 100` cũng làm mất phần lẻ (`76 × 67 // 100 = 50` thay vì `50.92`). **Sửa:** chia thực `/`.

### Lỗi 3 — Lấy giá theo **vị trí** `prices[mã]` thay vì theo mã

In [23]:
class WrongIndexPrice(CashierDict):
    def __init__(self, n, discount, products, prices):
        super().__init__(n, discount, products, prices)
        self.prices = prices
    def getBill(self, product, amount):
        self.count += 1
        bill = 0
        for p, a in zip(product, amount):
            if p - 1 < len(self.prices):                 # tránh IndexError để thấy kết quả sai
                bill += a * self.prices[p - 1]           # giả định mã = vị trí + 1
        return bill * (100 - self.discount) / 100 if self.count % self.n == 0 else float(bill)

ops = ["Cashier", "getBill"]
args = [[5, 0, [10, 3], [70, 20]], [[3, 10], [1, 1]]]
print("sai :", run(WrongIndexPrice, ops, args), "| đúng:", run(Cashier, ops, args))

sai : [None, 0.0] | đúng: [None, 90.0]


**Vì sao sai:** mã sản phẩm không phải chỉ số mảng (mã `10` nằm ở vị trí 0). Ví dụ đề tình cờ có `products = [1..7]` nên dễ bị lừa. **Sửa:** dựng bảng `mã → giá`.

### Lỗi 4 — Tạo lại bộ đếm trong `getBill` (biến cục bộ)

In [24]:
class WrongLocalCount(CashierDict):
    def getBill(self, product, amount):
        count = 0                                  # biến cục bộ, mất sau mỗi lần gọi
        count += 1
        bill = sum(a * self.price[p] for p, a in zip(product, amount))
        return bill * (100 - self.discount) / 100 if count % self.n == 0 else float(bill)

print("sai :", run(WrongLocalCount, EX_OPS, EX_ARGS))
print("đúng:", EX_OUT)

sai : [None, 500.0, 4000.0, 1600.0, 4000.0, 4000.0, 14700.0, 2500.0]
đúng: [None, 500.0, 4000.0, 800.0, 4000.0, 4000.0, 7350.0, 2500.0]


**Vì sao sai:** trạng thái phải lưu trong `self`, không phải biến cục bộ. Ở ví dụ, không ai được giảm. **Sửa:** `self.count` khởi tạo trong `__init__`.

### Lỗi 5 — Tính hệ số giảm sai chiều: `bill * discount / 100`

In [25]:
class WrongFactor(CashierDict):
    def getBill(self, product, amount):
        self.count += 1
        bill = sum(a * self.price[p] for p, a in zip(product, amount))
        return bill * self.discount / 100 if self.count % self.n == 0 else float(bill)   # là SỐ TIỀN GIẢM

ops = ["Cashier", "getBill"]
args = [[1, 20, [1], [100]], [[1], [1]]]
print("sai :", run(WrongFactor, ops, args), "| đúng:", run(Cashier, ops, args))

sai : [None, 20.0] | đúng: [None, 80.0]


**Vì sao sai:** `bill × discount/100` là **số tiền được giảm**, không phải số tiền phải trả (với 50% thì tình cờ trùng nhau — ví dụ đề không phát hiện được!). **Sửa:** `bill × (100 − discount)/100`. Bài học: luôn test thêm `discount ≠ 50`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Hai mảng song song "mã – giá trị" ⇒ `dict(zip(...))` ngay trong `__init__`.
- "Mỗi lần thứ `n`" ⇒ bộ đếm + `% n` (tăng trước, kiểm tra sau) hoặc bộ đếm vòng reset.
- Kết quả số thực có sai số cho phép ⇒ tính bằng `float` là đủ; muốn chắc thì kiểm chứng bằng `Fraction`.
- Ví dụ đề có giá trị "đặc biệt" (`discount = 50`, mã = 1..7) có thể che giấu lỗi — tự thêm test với giá trị lệch.
- Bài design dễ: tập trung vào **trạng thái nào cần lưu** và **tiền xử lý gì một lần**.

## 15. Biến thể của bài

1. **Nhiều mức giảm giá** theo các chu kỳ khác nhau (ví dụ mỗi khách thứ 3 giảm 10%, thứ 5 giảm 20%, lấy mức cao nhất) — Bài 5.
2. **Giá thay đổi theo thời gian**: thêm phương thức `updatePrice(id, price)` — dict cập nhật `O(1)`.
3. **Giảm giá theo ngưỡng hóa đơn** (trên X đồng giảm Y%) thay vì theo thứ tự khách.
4. **Làm tròn tới đơn vị tiền tệ** (ví dụ 2 chữ số thập phân) — dùng `Fraction`/`Decimal` để làm tròn đúng.
5. **LeetCode 1352, 1348**: các bài design khác cùng đợt — luyện chung kỹ năng lưu trạng thái + bộ giả lập.

## 16. So sánh các cách

| Cách | Ý tưởng | getBill | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tìm tuyến tính | duyệt `products` mỗi mục | `O(m·P)` | `O(P)` | baseline |
| 2. Dict + `% n` | bảng băm mã→giá | `O(m)` | `O(P)` | **nộp bài**, dễ đọc |
| 3. Mảng 201 + reset | chỉ số trực tiếp, hệ số tính sẵn | `O(m)` | `O(201)` | khi mã nhỏ, muốn nhanh nhất |
| 4. Fraction | số hữu tỉ chính xác | `O(m)` (+ phân số) | `O(P)` | kiểm chứng, làm tròn tiền tệ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Cashier(2, 10, [1, 2], [50, 30])`, rồi `getBill([1], [2])`, `getBill([2], [5])`, `getBill([1, 2], [1, 1])`. Ba kết quả là gì?

**Bài 2 (tính tay):** Với `n = 1, discount = 33`, bảng giá `{4: 25, 9: 8}`, hóa đơn `product = [9, 4], amount = [3, 2]`. Tính tạm tính và số tiền phải trả (giá trị chính xác dạng phân số).

**Bài 3 (điền TODO):** Hoàn thiện `build_price_table(products, prices)` trả về dict và **kiểm tra** mã trùng (raise `ValueError` nếu trùng).

**Bài 4 (tự viết lại):** Viết lại lớp `Cashier` dùng bộ đếm vòng (reset khi đạt `n`) và **dict**, không nhìn code mẫu.

**Bài 5 (mini challenge):** Lớp `CashierTiered(rules, products, prices)` với `rules` là list `(n_i, d_i)`: khách thứ `k` được mức giảm **lớn nhất** trong các `d_i` thỏa `k % n_i == 0` (không thỏa luật nào thì không giảm).

In [26]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def build_price_table(products, prices):
    table = {}
    # TODO: duyệt zip(products, prices); nếu mã đã có trong table -> raise ValueError
    # TODO: gán table[mã] = giá
    return table

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
class MyCashier:
    def __init__(self, n, discount, products, prices):
        pass  # TODO
    def getBill(self, product, amount):
        pass  # TODO

class CashierTieredTodo:
    def __init__(self, rules, products, prices):
        pass  # TODO
    def getBill(self, product, amount):
        pass  # TODO: d = max((d_i for n_i, d_i in rules if k % n_i == 0), default=0)

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Khách 2 được giảm. (2) Tạm tính: 100, 150, 80. (3) Khách 2 trả `150 × 0.9`.

**Bài 2:** (1) Tạm tính `3×8 + 2×25`. (2) `n = 1` ⇒ được giảm. (3) `74 × 67/100 = 4958/100 = 2479/50`.

**Bài 3:** (1) `for p, c in zip(products, prices)`. (2) `if p in table: raise ValueError(...)`. (3) Thử với mã trùng bằng `try/except`.

**Bài 4:** (1) `self.count = 0` trong `__init__`. (2) Tăng, nếu bằng `n` thì reset và giảm giá. (3) Đối chiếu ngẫu nhiên với `CashierExact`.

**Bài 5:** (1) Bộ đếm `k` toàn cục (không reset vì nhiều chu kỳ). (2) `max(..., default=0)`. (3) Kiểm chứng bằng bản `Fraction` viết riêng.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
ops = ["Cashier", "getBill", "getBill", "getBill"]
args = [[2, 10, [1, 2], [50, 30]], [[1], [2]], [[2], [5]], [[1, 2], [1, 1]]]
out = run(Cashier, ops, args)
print("Bài 1:", out)
assert same(out, [None, 100.0, 135.0, 80.0])

# Bài 2
sub = 3 * 8 + 2 * 25
exact = Fraction(sub) * Fraction(100 - 33, 100)
print("Bài 2: tạm tính", sub, "-> trả", exact, "=", float(exact))
assert exact == Fraction(2479, 50)
assert abs(run(Cashier, ["Cashier", "getBill"], [[1, 33, [4, 9], [25, 8]], [[9, 4], [3, 2]]])[1] - float(exact)) < 1e-9

Bài 1: [None, 100.0, 135.0, 80.0]
Bài 2: tạm tính 74 -> trả 2479/50 = 49.58


In [29]:
# Bài 3
def build_price_table_sol(products, prices):
    table = {}
    for p, c in zip(products, prices):
        if p in table:
            raise ValueError(f"mã trùng: {p}")
        table[p] = c
    return table

assert build_price_table_sol([1, 7], [5, 9]) == {1: 5, 7: 9}
try:
    build_price_table_sol([1, 1], [5, 9])
    raised = False
except ValueError as e:
    raised = True
    print("Bài 3: bắt được lỗi ->", e)
assert raised

# Bài 4
class MyCashierSol:
    def __init__(self, n, discount, products, prices):
        self.n, self.d = n, discount
        self.price = dict(zip(products, prices))
        self.count = 0
    def getBill(self, product, amount):
        bill = sum(a * self.price[p] for p, a in zip(product, amount))
        self.count += 1
        if self.count == self.n:
            self.count = 0
            return bill * (100 - self.d) / 100
        return float(bill)

rng = random.Random(4)
for _ in range(200):
    ops, args = random_case(rng, 15)
    ref = run(CashierExact, ops, args)
    out = run(MyCashierSol, ops, args)
    assert all(abs(Fraction(a) - b) <= Fraction(1, 10**5) for a, b in zip(out[1:], ref[1:]))
print("Bài 4: khớp bản chính xác trên 200 kịch bản")

Bài 3: bắt được lỗi -> mã trùng: 1
Bài 4: khớp bản chính xác trên 200 kịch bản


In [30]:
# Bài 5
class CashierTiered:
    def __init__(self, rules, products, prices):
        self.rules = rules
        self.price = dict(zip(products, prices))
        self.k = 0
    def getBill(self, product, amount):
        self.k += 1
        bill = sum(a * self.price[p] for p, a in zip(product, amount))
        d = max((d_i for n_i, d_i in self.rules if self.k % n_i == 0), default=0)
        return bill * (100 - d) / 100

class CashierTieredExact:
    def __init__(self, rules, products, prices):
        self.rules, self.products, self.prices, self.k = rules, products, prices, 0
    def getBill(self, product, amount):
        self.k += 1
        bill = Fraction(0)
        for p, a in zip(product, amount):
            bill += a * self.prices[self.products.index(p)]
        best = 0
        for n_i, d_i in self.rules:
            if self.k % n_i == 0 and d_i > best:
                best = d_i
        return bill * Fraction(100 - best, 100)

ops = ["CashierTiered"] + ["getBill"] * 6
args = [[[(3, 10), (5, 20)], [1], [100]]] + [[[1], [1]]] * 6
print("Bài 5:", run(CashierTiered, ops, args))
assert same(run(CashierTiered, ops, args), [None, 100.0, 100.0, 90.0, 100.0, 80.0, 90.0])
rng = random.Random(5)
for _ in range(200):
    ops, args = random_case(rng, 20)
    rules = [(rng.randint(1, 6), rng.randint(0, 100)) for _ in range(rng.randint(1, 3))]
    args[0] = [rules, args[0][2], args[0][3]]
    out, ref = run(CashierTiered, ops, args), run(CashierTieredExact, ops, args)
    assert all(abs(Fraction(a) - b) <= Fraction(1, 10**5) for a, b in zip(out[1:], ref[1:]))
print("Bài 5: khớp bản chính xác trên 200 kịch bản")

Bài 5: [None, 100.0, 100.0, 90.0, 100.0, 80.0, 90.0]


Bài 5: khớp bản chính xác trên 200 kịch bản


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
class Cashier:
    def __init__(self, n, discount, products, prices):
        self.n, self.discount = n, discount
        self.price = dict(zip(products, prices))
        self.count = 0
    def getBill(self, product, amount):
        self.count += 1                                   # tăng TRƯỚC
        bill = sum(a * self.price[p] for p, a in zip(product, amount))
        if self.count % self.n == 0:
            return bill * (100 - self.discount) / 100     # chia THỰC
        return float(bill)
```

**Checklist trước khi submit:**
- [ ] Bộ đếm lưu trong `self`, tăng **trước** khi kiểm tra.
- [ ] Hệ số là `(100 − discount)/100`, không phải `discount/100`.
- [ ] Chia thực `/`, không `//`.
- [ ] Tra giá theo **mã** (dict), không theo vị trí.
- [ ] Test thêm `discount ≠ 50`, `n = 1`, `discount = 0/100`.
- [ ] Đúng chữ ký: `Cashier(n, discount, products, prices)`, `getBill(product, amount) -> float`.